# Advanced EDA: CineMind AI
Explore MovieLens behavior, segmentation signals, and recommendation insights.

**Key sections:**
- Ratings distribution and user activity
- Genre popularity and correlation heatmap
- Movie popularity analysis
- KMeans clusters with PCA visualization

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("..") / "data"
movies = pd.read_csv(DATA_DIR / "movies.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")
data = ratings.merge(movies, on="movieId", how="left")

data.head()

In [ ]:
data.info()

data.describe()

data.sample(5, random_state=42)

In [ ]:
user_activity = data.groupby("userId").agg(ratings_count=("rating", "count"))
movie_popularity = data.groupby("title").agg(rating_count=("rating", "count"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(ratings["rating"], bins=10, ax=axes[0], color="#f44d4d")
axes[0].set_title("Ratings Distribution")
sns.histplot(user_activity["ratings_count"], bins=30, ax=axes[1], color="#5fa8d3")
axes[1].set_title("User Activity (Ratings Count)")
plt.tight_layout()
plt.show()

top_movies = movie_popularity.sort_values("rating_count", ascending=False).head(15)
plt.figure(figsize=(12, 4))
sns.barplot(x=top_movies.index, y=top_movies["rating_count"], color="#f44d4d")
plt.xticks(rotation=45, ha="right")
plt.title("Most Rated Movies")
plt.show()

In [ ]:
genre_counts = movies["genres"].str.split("|").explode().value_counts()
top_genres = genre_counts.head(12)

plt.figure(figsize=(12, 4))
sns.barplot(x=top_genres.index, y=top_genres.values, color="#5fa8d3")
plt.xticks(rotation=45, ha="right")
plt.title("Top Genres")
plt.show()

genre_data = data.assign(genres=data["genres"].str.split("|")).explode("genres")
genre_data = genre_data[genre_data["genres"].isin(top_genres.index)]
genre_user = genre_data.pivot_table(index="userId", columns="genres", values="rating", aggfunc="mean")
plt.figure(figsize=(10, 6))
sns.heatmap(genre_user.corr(), cmap="coolwarm", center=0)
plt.title("Genre Correlation Heatmap")
plt.show()

In [ ]:
user_movie_matrix = data.pivot_table(index="userId", columns="title", values="rating").fillna(0)
scaler = StandardScaler()
scaled = scaler.fit_transform(user_movie_matrix)

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
clusters = kmeans.fit_predict(scaled)

pca = PCA(n_components=2, random_state=42)
reduced = pca.fit_transform(scaled)

plt.figure(figsize=(8, 6))
plt.scatter(reduced[:, 0], reduced[:, 1], c=clusters, cmap="tab10", alpha=0.7)
plt.title("PCA Cluster Visualization")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

cluster_summary = (
    pd.DataFrame({"userId": user_movie_matrix.index, "cluster": clusters})
    .merge(data[["userId", "genres"]], on="userId", how="left")
    .assign(genres=lambda df: df["genres"].str.split("|"))
    .explode("genres")
    .groupby(["cluster", "genres"]).size().reset_index(name="count")
)
cluster_summary.sort_values(["cluster", "count"], ascending=[True, False]).head(20)